### Overview
This notebook performs advanced data cleaning and normalization on the movie dataset created by Sprint1_Raw.ipynb. It focuses on handling missing values, standardizing data types, and cleaning genre information using fuzzy matching techniques.

In [ ]:
import numpy as np
import pandas as pd
from thefuzz import fuzz

In [ ]:
df = pd.read_parquet("Films.parquet")

In [ ]:
df

In [ ]:
# Convert year to datetime format, filling unparseable values with default year 1995
df["year"] = pd.to_datetime(df["year"], errors='coerce')

mask = df["year"].isna()

df.loc[mask, "year"] = pd.to_datetime("1995")

In [ ]:
df

In [ ]:
# Fill missing movie titles with "Phantom of the Opera, The" (MovieID 1361 - known data quality issue)
mask = df["title"].isna() 
df.loc[mask, "title"] = "Phantom of the Opera, The"

In [ ]:
# Find genre entries with hyphens that are NOT valid hyphenated genres (Sci-Fi, Film-Noir)
# This identifies malformed genre data like "Comedy-Horror" that should be ["Comedy", "Horror"]
mask = (df["genres"].str[0].str.contains("-")) & ~(df["genres"].str[0].str.contains("Sci-Fi")) & ~(df["genres"].str[0].str.contains("Film-Noir"))

pd.set_option('display.max_rows', 500)
df.loc[mask]

In [ ]:
# corrected malfored row
df.at[df.loc[mask].index[0], "genres"] = ["Comedy", "Horror"]

In [ ]:
df[mask]

In [ ]:
# identified entries similar to drama but misspelled
mask = df["genres"].str[0].apply(lambda x: (fuzz.ratio(x, "Drama") >= 50 and x != "Drama"))
df.loc[mask]

In [ ]:
# replace all misspelled values with "Drama"
# create a numpy array for each mistake identified with the mask
drama_values = [np.array(["Drama"]) for _ in range(5)]

df.loc[mask, "genres"] = drama_values

In [ ]:
# normalization function to handle all possible data types in genres column
# Ensures every value becomes a Python list
def normalize_to_list(val):
    
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return []

    
    if isinstance(val, np.ndarray):
        return val.tolist()

    
    if isinstance(val, list):
        return val

    
    if isinstance(val, str):
        return [val]

    
    return [val]

df["genres"] = df["genres"].apply(normalize_to_list)

In [ ]:
print(df["genres"].apply(type).value_counts())

In [ ]:
# create parquet
df.to_parquet("Movies_Cleaned_frfr.parquet")
